# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [ ]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 177MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=1a24f91d-ad8a-4068-884f-f57fb4c89152
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 109MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 141MB/s]


In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [ ]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [ ]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def random_start(x, eps, attack="pgd_linf"):
    # Returns a perturbation delta with delta <= eps and x + delta in [0, 1].
    if attack == "pgd_linf":
        delta = torch.empty_like(x).uniform_(-eps, eps)
    elif attack == "pgd_l2":
        delta = torch.empty_like(x).normal_()
        n = l2_norm(delta)
        r = torch.zeros_like(n).uniform_(0, 1)
        delta = delta * (r / n * eps)
    else:
        raise ValueError(f"unknown attack: {attack}")
    return clip_box(x, delta).detach()


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    x = x.detach()
    delta = random_start(x, eps, "pgd_linf") if rand_init else torch.zeros_like(x)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)  # L-inf projection
        delta = clip_box(x, delta)                              # image-domain clip

    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    x = x.detach()
    tiny = 1e-10
    delta = random_start(x, eps, "pgd_l2") if rand_init else torch.zeros_like(x)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        grad = grad / (l2_norm(grad) + tiny)
        delta = delta + alpha * grad
        delta = delta * (eps / (l2_norm(delta) + tiny)).clamp(max=1.0)  # L2 ball projection
        delta = clip_box(x, delta)                                      # image-domain clip

    return (x + delta).detach()

In [ ]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [ ]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
res_base = test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (118 s)


# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [ ]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [ ]:
def single_norm_loss(model, x, y, norm):
    # Attack in eval mode (BatchNorm statistics unchanged), then compute the loss in train mode.
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
res_linf = test_model(model_linf)

epoch 1: loss 1.203 (42 s)
epoch 2: loss 1.180 (83 s)
epoch 3: loss 1.151 (124 s)
epoch 4: loss 1.126 (165 s)
epoch 5: loss 1.097 (206 s)
epoch 6: loss 1.068 (247 s)
epoch 7: loss 1.049 (288 s)
epoch 8: loss 1.030 (329 s)
epoch 9: loss 1.013 (370 s)
epoch 10: loss 1.007 (411 s)
clean 78.3  linf 44.8  l2 57.0  l1 20.2  union 20.0   (124 s)


In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
res_l2 = test_model(model_l2)

epoch 1: loss 0.897 (41 s)
epoch 2: loss 0.826 (82 s)
epoch 3: loss 0.787 (123 s)
epoch 4: loss 0.754 (164 s)
epoch 5: loss 0.714 (205 s)
epoch 6: loss 0.674 (246 s)
epoch 7: loss 0.631 (287 s)
epoch 8: loss 0.609 (328 s)
epoch 9: loss 0.591 (369 s)
epoch 10: loss 0.583 (410 s)
clean 82.3  linf 32.2  l2 60.2  l1 28.2  union 24.3   (124 s)


In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
res_l1 = test_model(model_l1)

epoch 1: loss 1.231 (42 s)
epoch 2: loss 0.904 (84 s)
epoch 3: loss 0.606 (126 s)
epoch 4: loss 0.487 (168 s)
epoch 5: loss 0.356 (210 s)
epoch 6: loss 0.278 (252 s)
epoch 7: loss 0.225 (294 s)
epoch 8: loss 0.190 (335 s)
epoch 9: loss 0.168 (377 s)
epoch 10: loss 0.165 (419 s)
clean 84.3  linf 0.0  l2 0.1  l1 26.8  union 0.0   (124 s)


In [ ]:
# Part (a) table: base model and the three single-norm models
results_a = {"base": res_base, "linf-AT": res_linf, "l2-AT": res_l2, "l1-AT": res_l1}
print(f"{'model':8s}" + "".join(f"{k:>8s}" for k in ("clean", *NORMS, "union")))
for name, r in results_a.items():
    print(f"{name:8s}" + "".join(f"{r[k]:8.1f}" for k in ("clean", *NORMS, "union")))

model      clean    linf      l2      l1   union
base        78.7    48.4    59.8    22.4    22.3
linf-AT     78.3    44.8    57.0    20.2    20.0
l2-AT       82.3    32.2    60.2    28.2    24.3
l1-AT       84.3     0.0     0.1    26.8     0.0


### Part (a) answers

| Model | Clean | ℓ∞ | ℓ2 | ℓ1 | Union |
|---|---|---|---|---|---|
| Base | 78.7 | 48.4 | 59.8 | 22.4 | 22.3 |
| ℓ∞-AT | 78.3 | 44.8 | 57.0 | 20.2 | 20.0 |
| ℓ2-AT | 82.3 | 32.2 | 60.2 | 28.2 | 24.3 |
| ℓ1-AT | 84.3 | 0.0 | 0.1 | 26.8 | 0.0 |

**(i) Weakest norm.**
- Base and ℓ∞-AT: ℓ1 (22.4 and 20.2).
- ℓ2-AT: ℓ1 (28.2), then ℓ∞ (32.2).
- ℓ1-AT: ℓ∞ and ℓ2 (0.0 and 0.1).

For the first three models the union accuracy is at or below the weakest norm, so ℓ1 is what limits them.

**(ii) Does robustness transfer?** Only partly, and not evenly.
- ℓ∞ and ℓ2 help each other: ℓ∞-AT keeps ℓ2 at 57.0, and ℓ2-AT keeps ℓ∞ at 32.2 (but loses 16 points of the base's 48.4).
- ℓ1 is the odd one out. Training on ℓ∞ does not help it (22.4 → 20.2), and ℓ2 training helps a bit (22.4 → 28.2).
- Training on ℓ1 transfers nothing the other way: ℓ∞ and ℓ2 fall from 48.4 and 59.8 to 0.0 and 0.1. It also pushes clean accuracy up (84.3), which suggests the model gave up its ℓ∞/ℓ2 robustness. Its ℓ1 accuracy is only 26.8 even though its training loss ended near 0.17, so the 10-step training attack looks much weaker than the 50-step test attack.
- Fine-tuning on the 10k subset also costs the base some ℓ∞ accuracy (48.4 → 44.8 for ℓ∞-AT).

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    # q = linf, r = l1. Attack in eval mode, compute the losses in train mode.
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q, logits_r = model(x_q), model(x_r)

    # mean over the batch of max(CE(x_q), CE(x_r))
    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    loss_adv = torch.maximum(ce_q, ce_r).mean()

    # KL(p_q || p_r), only on images classified correctly under the l_q (linf) attack
    kl = F.kl_div(F.log_softmax(logits_r, dim=1), F.softmax(logits_q, dim=1), reduction="none").sum(1)
    ok = (logits_q.argmax(1) == y).float()
    loss_kl = (kl * ok).sum() / ok.sum().clamp_min(1.0)

    return loss_adv + lam * loss_kl


def gradient_projection(g_n, g_a):
    # cos(g_n, g_a) * g_n if cos > 0, else 0 (for one parameter tensor)
    cos = (g_n * g_a).sum() / (g_n.norm() * g_a.norm() + 1e-12)
    return cos * g_n if cos > 0 else torch.zeros_like(g_n)

In [ ]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
res_ramp = test_model(model_ramp)

epoch 1: loss 1.432 (89 s)
epoch 2: loss 1.330 (178 s)
epoch 3: loss 1.303 (266 s)
epoch 4: loss 1.281 (355 s)
epoch 5: loss 1.253 (444 s)
epoch 6: loss 1.227 (532 s)
epoch 7: loss 1.199 (621 s)
epoch 8: loss 1.186 (710 s)
epoch 9: loss 1.171 (799 s)
epoch 10: loss 1.165 (887 s)
clean 79.3  linf 41.0  l2 61.9  l1 44.6  union 38.8   (124 s)


### Part (b) answers

Time per epoch is the total printed training time divided by the 10 epochs (the printed value is cumulative).

| Model | Clean | ℓ∞ | ℓ2 | ℓ1 | Union | s / epoch |
|---|---|---|---|---|---|---|
| Base | 78.7 | 48.4 | 59.8 | 22.4 | 22.3 | – |
| ℓ∞-AT | 78.3 | 44.8 | 57.0 | 20.2 | 20.0 | 41.1 |
| ℓ2-AT | 82.3 | 32.2 | 60.2 | 28.2 | 24.3 | 41.0 |
| ℓ1-AT | 84.3 | 0.0 | 0.1 | 26.8 | 0.0 | 41.9 |
| RAMP (seed 0) | 79.3 | 41.0 | 61.9 | 44.6 | 38.8 | 88.7 |

**(i)** Highest union accuracy: RAMP (38.8), far above the best single-norm model (ℓ2-AT, 24.3). Highest clean accuracy: ℓ1-AT (84.3), then ℓ2-AT (82.3); RAMP (79.3) is close to the base (78.7). RAMP gets the union gain mainly from ℓ1 (22.4 → 44.6) and pays with ℓ∞ (48.4 → 41.0).

**(ii)** RAMP takes about 2.2× the time of single-norm training (88.7 s vs 41.1 s per epoch). Each RAMP step runs two attacks (ℓ∞ and ℓ1, 10 steps each) instead of one, and the attack is most of the cost of a step. It also does forward passes on two adversarial batches, plus an extra forward and a second backward pass on the clean batch to get g_n for the gradient projection.

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [ ]:
# Weighted loss over linf and l1: the norm that currently has the higher loss gets more weight.
# MY_CFG: tau = how strongly the weights favour the weaker norm (0 = equal weights),
# window = number of recent batches in the moving average of each norm's loss.
from collections import deque

MY_CFG = {"norms": ("linf", "l1"), "tau": 2.0, "window": 20}
MY_STATE = {"hist": {}, "w_log": []}   # last window batch losses of each norm, and the weights used in every batch


def my_loss(model, x, y):
    norms = MY_CFG["norms"]
    model.eval()
    x_adv = {n: attack(model, x, y, n) for n in norms}
    model.train()
    losses = [F.cross_entropy(model(x_adv[n]), y) for n in norms]

    # Simple moving average of each norm's loss over the last window
    avg = []
    for n, l in zip(norms, losses):
        h = MY_STATE["hist"].setdefault(n, deque(maxlen=MY_CFG["window"]))
        h.append(l.item())
        avg.append(sum(h) / len(h))

    w = torch.softmax(MY_CFG["tau"] * torch.tensor(avg, device=x.device), dim=0)
    MY_STATE["w_log"].append(w.tolist())
    return sum(w[k] * losses[k] for k in range(len(norms)))

### Part (c): Focus on the worst norm (weighted loss over ℓ∞ and ℓ1)

**How I got to the idea.**
1. The union accuracy counts an image as failed if even one of the three attacks breaks the model. So the model is only as good as its weakest norm.
2. But when training on an average of the norms, nothing tells the model which norm is the one failing. We need a signal for that.
3. The adversarial loss on each norm is that signal: a higher loss on a norm means the model is currently worse there.
4. So combine the per-norm losses with weights that favour the worse norm, and the updates keep pushing on the weakest frontier. In every batch:
   - attack the batch with ℓ∞ and with ℓ1, and compute the cross-entropy loss on each (loss_linf, loss_l1);
   - for each norm, take the average of its loss over the last 20 batches (a simple moving average, so one noisy batch does not flip the weights);
   - weights = softmax(tau × these two averages), with tau = 2, so the two weights add up to 1 and the norm with the higher average loss gets the larger weight;
   - total loss = w_linf × loss_linf + w_l1 × loss_l1.

   The weights are treated as constants in the backward pass. tau only sets how strongly the weights favour the weaker norm. With tau = 0 both norms get weight 0.5 (a plain average). A larger tau puts more weight on the norm with the higher loss.
5. I attack only with ℓ∞ and ℓ1, which are the two most different regions (a small change on every pixel vs. a large change on a few).

In [ ]:
# Part (c): RAMP with seeds 0 and 1. adv_train prints the time per epoch.
# The RAMP run of part (b) already used the default seed=0 with the same settings, so it is reused as the seed-0 RAMP run.
def mean_of(rs):
    return {k: sum(r[k] for r in rs) / len(rs) for k in rs[0]}

runs = {"mine": {}, "ramp": {0: res_ramp}}
runs["ramp"][1] = test_model(adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1))

epoch 1: loss 1.434 (89 s)
epoch 2: loss 1.333 (177 s)
epoch 3: loss 1.299 (266 s)
epoch 4: loss 1.274 (355 s)
epoch 5: loss 1.248 (444 s)
epoch 6: loss 1.221 (533 s)
epoch 7: loss 1.201 (621 s)
epoch 8: loss 1.186 (710 s)
epoch 9: loss 1.178 (799 s)
epoch 10: loss 1.169 (887 s)
clean 79.2  linf 42.5  l2 62.6  l1 46.8  union 41.0   (124 s)


In [ ]:
# Part (c): my method, seeds 0 and 1. The models are kept, so the strong evaluation below needs no retraining.
models_mine = {}
runs["mine"] = {}
for seed in (0, 1):
    MY_STATE["hist"], MY_STATE["w_log"] = {}, []
    models_mine[seed] = adv_train(load_base_model(), my_loss, seed=seed)
    runs["mine"][seed] = test_model(models_mine[seed])
    # mean weight of each norm in each epoch (columns: the norms in MY_CFG["norms"])
    print("weights per epoch", MY_CFG["norms"]); print(torch.tensor(MY_STATE["w_log"]).view(10, -1, 2).mean(1).numpy().round(2))

base = res_base
print(f"{'model':8s}" + "".join(f"{k:>8s}" for k in ("clean", *NORMS, "union")))
for name, r in (("base", base), ("ramp", mean_of(list(runs["ramp"].values()))), ("mine", mean_of(list(runs["mine"].values())))):
    print(f"{name:8s}" + "".join(f"{r[k]:8.1f}" for k in ("clean", *NORMS, "union")))

epoch 1: loss 1.316 (83 s)
epoch 2: loss 1.214 (165 s)
epoch 3: loss 1.185 (248 s)
epoch 4: loss 1.156 (331 s)
epoch 5: loss 1.125 (413 s)
epoch 6: loss 1.095 (496 s)
epoch 7: loss 1.063 (579 s)
epoch 8: loss 1.047 (661 s)
epoch 9: loss 1.030 (744 s)
epoch 10: loss 1.025 (827 s)
clean 77.8  linf 41.0  l2 62.0  l1 47.4  union 39.6   (124 s)
weights per epoch ('linf', 'l1')
[[0.47 0.53]
 [0.54 0.46]
 [0.55 0.45]
 [0.56 0.44]
 [0.56 0.44]
 [0.57 0.43]
 [0.57 0.43]
 [0.57 0.43]
 [0.57 0.43]
 [0.57 0.43]]
epoch 1: loss 1.316 (83 s)
epoch 2: loss 1.218 (165 s)
epoch 3: loss 1.181 (248 s)
epoch 4: loss 1.148 (330 s)
epoch 5: loss 1.119 (413 s)
epoch 6: loss 1.089 (496 s)
epoch 7: loss 1.064 (578 s)
epoch 8: loss 1.047 (661 s)
epoch 9: loss 1.035 (744 s)
epoch 10: loss 1.025 (827 s)
clean 77.9  linf 42.1  l2 62.3  l1 49.3  union 40.9   (124 s)
weights per epoch ('linf', 'l1')
[[0.47 0.53]
 [0.54 0.46]
 [0.55 0.45]
 [0.56 0.44]
 [0.57 0.43]
 [0.57 0.43]
 [0.57 0.43]
 [0.57 0.43]
 [0.57 0.43]
 [

**Results (mean of seeds 0 and 1)**

| Model | Clean | ℓ∞ | ℓ2 | ℓ1 | Union | s / epoch |
|---|---|---|---|---|---|---|
| Base | 78.7 | 48.4 | 59.8 | 22.4 | 22.3 | – |
| RAMP | 79.2 | 41.8 | 62.2 | 45.7 | 39.9 | 88.7 |
| Mine | 77.8 | 41.5 | 62.1 | 48.3 | 40.2 | 82.7 |

Per seed (union): mine 39.6 / 40.9, RAMP 38.8 / 41.0.

Time per epoch: mine takes 82.7 s and RAMP 88.7 s.

In [ ]:
# Ablation 1: uniform weights (tau = 0, i.e. a plain average of the linf and l1 losses).
# With tau = 0 the weights are 0.5 / 0.5 whatever the moving average is, so this run does not depend on it.
ablations = {}
saved = dict(MY_CFG)
MY_CFG.update({"tau": 0.0})
MY_STATE["hist"] = {}
ablations["tau=0 (AVG)"] = test_model(adv_train(load_base_model(), my_loss, seed=0))
MY_CFG.clear(); MY_CFG.update(saved)

epoch 1: loss 1.300 (83 s)
epoch 2: loss 1.207 (165 s)
epoch 3: loss 1.174 (248 s)
epoch 4: loss 1.143 (331 s)
epoch 5: loss 1.112 (413 s)
epoch 6: loss 1.080 (496 s)
epoch 7: loss 1.046 (579 s)
epoch 8: loss 1.029 (661 s)
epoch 9: loss 1.012 (744 s)
epoch 10: loss 1.007 (826 s)
clean 78.1  linf 40.7  l2 61.7  l1 48.4  union 39.6   (124 s)


In [ ]:
# Ablation 2: also attack l2 in training.
saved = dict(MY_CFG)
MY_CFG.update({"norms": ("linf", "l2", "l1")})
MY_STATE["hist"], MY_STATE["w_log"] = {}, []
ablations["+l2"] = test_model(adv_train(load_base_model(), my_loss, seed=0))
print("weights per epoch", MY_CFG["norms"]); print(torch.tensor(MY_STATE["w_log"]).view(10, -1, 3).mean(1).numpy().round(2))
MY_CFG.clear(); MY_CFG.update(saved)

epoch 1: loss 1.253 (124 s)
epoch 2: loss 1.162 (247 s)
epoch 3: loss 1.124 (371 s)
epoch 4: loss 1.092 (495 s)
epoch 5: loss 1.067 (618 s)
epoch 6: loss 1.030 (742 s)
epoch 7: loss 1.010 (866 s)
epoch 8: loss 0.989 (989 s)
epoch 9: loss 0.972 (1113 s)
epoch 10: loss 0.966 (1237 s)
clean 78.9  linf 40.7  l2 62.0  l1 47.3  union 39.2   (124 s)
weights per epoch ('linf', 'l2', 'l1')
[[0.37 0.2  0.43]
 [0.42 0.21 0.37]
 [0.43 0.21 0.36]
 [0.44 0.21 0.35]
 [0.44 0.21 0.34]
 [0.44 0.21 0.34]
 [0.45 0.22 0.34]
 [0.45 0.22 0.34]
 [0.45 0.22 0.34]
 [0.45 0.22 0.34]]


**Ablation (seed 0).** I removed the adaptive weighting (τ = 0, i.e. a plain average of the ℓ∞ and ℓ1 losses), and separately added an ℓ2 attack. The τ = 0 run does not depend on the moving average (the weights are 0.5 / 0.5 either way).

| Variant | Clean | ℓ∞ | ℓ2 | ℓ1 | Union | s / epoch |
|---|---|---|---|---|---|---|
| Mine | 77.8 | 41.0 | 62.0 | 47.4 | 39.6 | 82.7 |
| τ = 0 (uniform average) | 78.1 | 40.7 | 61.7 | 48.4 | 39.6 | 82.6 |
| + ℓ2 attack | 78.9 | 40.7 | 62.0 | 47.3 | 39.2 | 123.7 |

**Analysis.**
- Versus the base model: union improves from 22.3 to 40.2, mostly through ℓ1 (22.4 →
48.3). ℓ∞ drops (48.4 → 41.5), clean accuracy slightly (78.7 → 77.8), and ℓ2 improves (59.8
→ 62.1) without being trained on.
- Versus RAMP: union is comparable (40.2 vs. 39.9), ahead in seed 0 and 0.1 behind in seed
1, and the gap is within the test-set noise (about ±1.5%). The method reaches the same
level with a simpler update (no KL term, no gradient projection) and about 7% less time per
epoch.
- Weights: in epoch 1, ℓ1 gets more weight (0.53), matching the base model’s weak ℓ1. From
epoch 2 on, ℓ∞ gets more (0.54, then 0.57 from epoch 6), as ℓ1 improves faster. Both seeds
show this pattern.
- Effect of weighting: none on the union. Uniform weights give 39.6, the same as with
weighting (39.6); the shift of accuracy toward ℓ∞ (ℓ∞ 40.7 → 41.0, ℓ1 48.4 → 47.4) cancels
in the union. Most of the gain comes from training on ℓ1 and ℓ∞ together.
- Adding ℓ2: no gain (union 39.2 vs. 39.6) at 1.5× the time per epoch. Its weight stayed
smallest (about 0.21), and ℓ2 robustness is obtained without training on it.
- Limitation: ℓ∞ is the weakest norm (41.5), and the union (40.2) sits just below it. A larger
τ could push harder on it; this was not tested.

In [ ]:
# Strong evaluation of my method on the model kept from the runs above.
best_seed = max(runs["mine"], key=lambda s: runs["mine"][s]["union"])   # the seed with the higher union above
print("best seed:", best_seed)
model_best = models_mine[best_seed]
res_strong = test_model(model_best, steps=200, restarts=5)

clean 77.9  linf 41.9  l2 62.3  l1 46.7  union 40.0   (1227 s)


**Evaluation check.** I evaluated the model of my method that scored the higher union (seed 1, 40.9 vs 39.6 for seed 0) again with stronger attacks (200 steps, 5 restarts; about 21 min, roughly 10× the default evaluation). It is the same model as in the table above, so the two rows are comparable.

| Evaluation | Clean | ℓ∞ | ℓ2 | ℓ1 | Union |
|---|---|---|---|---|---|
| Default (50 steps, 2 restarts) | 77.9 | 42.1 | 62.3 | 49.3 | 40.9 |
| Strong (200 steps, 5 restarts) | 77.9 | 41.9 | 62.3 | 46.7 | 40.0 |

The drops are small: 0.2 for ℓ∞, 0.0 for ℓ2, 2.6 for ℓ1 and 0.9 for the union. If the default evaluation were overestimating robustness through gradient masking, the stronger attacks would take away much more. ℓ1 is the norm that moves most.